# 03 — Feature Engineering & Selection

**Goal:** turn the cleaned table into model-ready features.

Two strictly separated transform types:

1. **Stateless row-wise features** (`src.feature_engineering.add_features`) —
   no fitted parameters, so they can run before the split and be replayed
   identically at inference.
2. **Fitted transforms** (scaling, one-hot encoding) — live inside a sklearn
   `ColumnTransformer`, fitted on training folds only. This is the structural
   guarantee against train/test leakage.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [2]:
from src.data_loader import load_data
from src.preprocessing import clean_raw_data, split_data
from src.feature_engineering import add_features, build_preprocessor, get_feature_lists
from src.utils import load_config

config = load_config()
df = add_features(clean_raw_data(load_data(str(ROOT / config["data"]["raw_path"])), config))
df[["tenure_months", "monthly_charges", "total_charges",
    "num_addon_services", "avg_monthly_spend", "charge_growth", "tenure_bucket"]].head()

2026-07-31 09:39:34,691 | src.preprocessing | INFO | Cleaned data: 7043 rows, 20 columns


2026-07-31 09:39:34,697 | src.feature_engineering | INFO | Added engineered features: 24 total columns


,tenure_months,monthly_charges,total_charges,num_addon_services,avg_monthly_spend,charge_growth,tenure_bucket
0,2,53.85,108.15,2,54.075000,-0.225000,0-1yr
1,2,70.70,151.65,0,75.825000,-5.125000,0-1yr
2,8,99.65,820.50,3,102.562500,-2.912500,0-1yr
3,28,104.80,3046.05,4,108.787500,-3.987500,2-4yr
4,49,103.70,5036.30,4,102.781633,0.918367,4yr+


## Why these four engineered features?

| Feature | Hypothesis | EDA evidence |
|---|---|---|
| `num_addon_services` | ecosystem lock-in reduces churn | add-on users churn less |
| `avg_monthly_spend` | lifetime average bill ≈ price the customer accepted | — |
| `charge_growth` | current bill − lifetime average: a **recent price increase** triggers churn | churners skew to high bills |
| `tenure_bucket` | churn risk is non-linear in tenure (cliff in year 1) | early-life churn spike |

Each one is computable from a single customer row at prediction time — no
aggregates over other customers, so no leakage.

In [3]:
X_train, X_test, y_train, y_test = split_data(df, config)
numeric, categorical = get_feature_lists(X_train)
print(f"{len(numeric)} numeric: {numeric}")
print(f"{len(categorical)} categorical: {categorical}")

2026-07-31 09:39:34,727 | src.preprocessing | INFO | Split data: train=5634 rows (churn 26.5%), test=1409 rows (churn 26.5%)


6 numeric: ['tenure_months', 'monthly_charges', 'total_charges', 'num_addon_services', 'avg_monthly_spend', 'charge_growth']
17 categorical: ['gender', 'senior_citizen', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'tenure_bucket']


## The preprocessing pipeline

* **StandardScaler** on numerics — required by logistic regression, harmless for trees.
* **OneHotEncoder** on categoricals — `drop='if_binary'` avoids redundant dummy
  columns; `handle_unknown='ignore'` means an unseen category at inference
  encodes as all-zeros instead of crashing the service.

The preprocessor is **fitted here on the training split only** — fitting it on
all 7,043 rows would let test-set statistics (means, variances) leak into
training.

In [4]:
preprocessor = build_preprocessor(numeric, categorical)
Xt = preprocessor.fit_transform(X_train)
print(f"input:  {X_train.shape[1]} features")
print(f"output: {Xt.shape[1]} model-ready columns")
list(preprocessor.get_feature_names_out())

input:  23 features
output: 33 model-ready columns


['tenure_months',
 'monthly_charges',
 'total_charges',
 'num_addon_services',
 'avg_monthly_spend',
 'charge_growth',
 'gender_Male',
 'senior_citizen_Yes',
 'partner_Yes',
 'dependents_Yes',
 'phone_service_Yes',
 'multiple_lines_Yes',
 'internet_service_DSL',
 'internet_service_Fiber optic',
 'internet_service_No',
 'online_security_Yes',
 'online_backup_Yes',
 'device_protection_Yes',
 'tech_support_Yes',
 'streaming_tv_Yes',
 'streaming_movies_Yes',
 'contract_Month-to-month',
 'contract_One year',
 'contract_Two year',
 'paperless_billing_Yes',
 'payment_method_Bank transfer (automatic)',
 'payment_method_Credit card (automatic)',
 'payment_method_Electronic check',
 'payment_method_Mailed check',
 'tenure_bucket_0-1yr',
 'tenure_bucket_1-2yr',
 'tenure_bucket_2-4yr',
 'tenure_bucket_4yr+']

## Feature selection: why we did NOT run an automated selector

With 23 input features (→ ~45 encoded columns) and 5,634 training rows, the
feature-to-sample ratio is comfortable. Selection already happened where it
matters — during **leakage detection** (dropped `Churn Score`, `CLTV`,
`Churn Reason`, `Churn Label`) and **information screening** (dropped
constants, IDs and geography). Beyond that, L2 regularisation (logistic) and
per-split feature sampling (trees/XGBoost) handle redundancy better than a
hard filter, and permutation importance in notebook 05 verifies nothing we
kept is dead weight.